In [ ]:
import torch
import equivit
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# `equivit.geometry`: group actions

## (A) Groups and Representations
Irreps of dihedral groups Dn and cyclic groups Cn are implemented for arbitrary n.

In [ ]:
n = 8
group = equivit.geometry.DihedralGroup(n) # this will be Dn (order=2n)

# group action is implemented by __mul__
print(group['r'] * group['r'])
print(group['r'] * group['t'])

In [ ]:
# we can get a list of (real) irreps
irreps = group.real_irreps()

print(group)
print(group.order())
print('----------------')

for irrep_name, irrep in irreps.items():
    print(f'name={irrep_name}, dimension={irrep.dim}')
    print(f'characters={irrep.characters()}') # characters of all conjugacy classes

In [ ]:
# show representation matrices for E1
print(irreps['E1'](group['t']))
print(irreps['E1'](group['r']))

## (B) Lattices - Group Actions

In [ ]:
# create a honeycomb lattice
honey = equivit.geometry.Honeycomb(3)

# by default, a honeycomb lattice comes with a D6-action
print(honey.action.group)

# we can compute the orbits
orbits = honey.action.orbits()

fig = plt.figure(figsize=(4,12))
axs = fig.subplots(1,3)

# visualize the 0th, 1st, and 2nd orbits
for i in [0,1,2]:
    axs[i].scatter(*honey.points.T, s=10, color='k')
    axs[i].scatter(*honey.points[orbits[i]].T, s=20, color='red')
    axs[i].set_aspect('equal')


# `equivit.nn`: Equivariant ViT Layers
Demo: constructing a D6-equivariant patch embedding layer and visualizing the projection tensors

### 1. Construct hexagonal lattice

In [ ]:
hexagon = equivit.geometry.Hexagon(5)


print(hexagon.L) # number of lattice sites (pixels)

# dimension of each irrep of D6, should be [1, 1, 1, 1, 2, 2]
irrep_dims = [irrep.dim for irrep in hexagon.action.group.real_irreps().values()] 
print(irrep_dims)

# visualize the hexagonal lattice
fig = plt.figure(figsize=(3,3))
ax = fig.add_subplot()
ax.scatter(*hexagon.points.T)
ax.set_aspect('equal')

### 2. Build EquivariantPatchEmbed module

In [ ]:
irrep_channels = [16,8,8,8,8,8] # 6 entries because D6 has 6 irreps (A1, A2, B1, B2, E1, E2)
patch_embed = equivit.nn.EquivariantPatchEmbed(hexagon, 
                                 in_channels=3, # 3 input channels
                                 out_channels=irrep_channels, 
                                 subgroup_args=('D', 6, 0), # specify that we want to use the full symetry group (D6)
                                )

### 3. Visualize the projections

In [ ]:
projections = patch_embed.get_projections() 
# this is a list of 6 tensors, each of shape (L*3, Ci*di)

for p in projections:
    print(p.shape)

# reshape each tensor to (L, 3, Ci, di)
reshaped_projections = [proj.reshape(hexagon.L, 3, irrep_channels[i], irrep_dims[i]) 
                        for i,proj in enumerate(projections)]

In [ ]:
def normalize(x):
    return (x - x.min()) / (x.max() - x.min())

fig = plt.figure()
axs = fig.subplots(1,4)

# visualize one A1 projection
# a Lattice object has a built-in method to visualize tensors
hexagon.colormesh(axs[0], normalize(reshaped_projections[0][:,0,0,0]), cmap=cm.coolwarm)

# visualize one A2 projection
hexagon.colormesh(axs[1], normalize(reshaped_projections[1][:,0,0,0]), cmap=cm.coolwarm)

# visiualize one component of one E1 projection
hexagon.colormesh(axs[2], normalize(reshaped_projections[4][:,0,0,0]), cmap=cm.coolwarm)

# visiualize one component of one E2 projection
hexagon.colormesh(axs[3], normalize(reshaped_projections[5][:,0,0,0]), cmap=cm.coolwarm)